[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/bases-de-datos-vectoriales/04-rag-avanzado.ipynb)

# RAG Avanzado: HyDE, Parent-Child y Self-Query

> Domina las estrategias avanzadas de RAG que van más allá de la búsqueda vectorial
> estándar: HyDE genera documentos hipotéticos, Parent-Child mejora el contexto
> y Self-Query traduce preguntas a filtros de metadatos.

## Objetivos
- Implementar HyDE (Hypothetical Document Embeddings)
- Implementar chunking Parent-Child para contexto más rico
- Implementar Self-Query para filtrado automático por metadatos
- Comparar las 3 estrategias en el mismo corpus

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic chromadb sentence-transformers --quiet

## 2. Corpus y configuración base

In [ ]:
import anthropic
import chromadb
from sentence_transformers import SentenceTransformer
import json

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print("Cargando modelo de embeddings...")
modelo_emb = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Corpus: artículos técnicos con párrafos padres e hijos
ARTICULOS_PADRE = [
    {"id": "art1", "titulo": "Machine Learning", "categoria": "ia", "nivel": "basico",
     "texto": "El machine learning es una rama de la inteligencia artificial donde los algoritmos aprenden patrones de datos sin ser programados explícitamente. Se divide en tres paradigmas: supervisado, no supervisado y por refuerzo. Los modelos supervisados aprenden de datos etiquetados para hacer predicciones. Los no supervisados encuentran estructura en datos sin etiquetar. El aprendizaje por refuerzo entrena agentes que maximizan recompensas en entornos dinámicos."},
    {"id": "art2", "titulo": "Redes Neuronales", "categoria": "ia", "nivel": "avanzado",
     "texto": "Las redes neuronales artificiales están compuestas por capas de neuronas interconectadas que procesan información de forma paralela. Cada neurona aplica una función de activación a la suma ponderada de sus entradas. El entrenamiento ajusta los pesos mediante backpropagation y gradient descent. Las arquitecturas modernas incluyen CNNs para imágenes, RNNs para secuencias y Transformers para lenguaje natural."},
    {"id": "art3", "titulo": "Bases de Datos Vectoriales", "categoria": "datos", "nivel": "intermedio",
     "texto": "Las bases de datos vectoriales almacenan y buscan embeddings de forma eficiente. Usan índices ANN (Approximate Nearest Neighbors) como HNSW o IVF para búsqueda rápida en millones de vectores. Los principales proveedores son Pinecone, Weaviate, Qdrant y ChromaDB. Son fundamentales en sistemas RAG, recomendaciones y búsqueda semántica."},
]

# Crear chunks hijos (fragmentos más pequeños de cada artículo padre)
chunks_hijos = []
for art in ARTICULOS_PADRE:
    oraciones = art["texto"].split(". ")
    for i, oracion in enumerate(oraciones):
        if oracion.strip():
            chunks_hijos.append({
                "id": f"{art['id']}_chunk{i}",
                "texto": oracion.strip() + ".",
                "padre_id": art["id"],
                "categoria": art["categoria"],
                "nivel": art["nivel"]
            })

# Indexar chunks hijos en ChromaDB
chroma = chromadb.EphemeralClient()
col_hijos = chroma.create_collection("chunks_hijos")
col_hijos.add(
    ids=[c["id"] for c in chunks_hijos],
    documents=[c["texto"] for c in chunks_hijos],
    embeddings=modelo_emb.encode([c["texto"] for c in chunks_hijos]).tolist(),
    metadatas=[{"padre_id": c["padre_id"], "categoria": c["categoria"], "nivel": c["nivel"]} for c in chunks_hijos]
)

# Indexar artículos padre también
col_padre = chroma.create_collection("articulos_padre")
col_padre.add(
    ids=[a["id"] for a in ARTICULOS_PADRE],
    documents=[a["texto"] for a in ARTICULOS_PADRE],
    embeddings=modelo_emb.encode([a["texto"] for a in ARTICULOS_PADRE]).tolist(),
    metadatas=[{"titulo": a["titulo"], "categoria": a["categoria"], "nivel": a["nivel"]} for a in ARTICULOS_PADRE]
)

print(f"✓ {len(ARTICULOS_PADRE)} artículos padre y {len(chunks_hijos)} chunks hijos indexados.")

## 3. Estrategia 1: HyDE (Hypothetical Document Embeddings)

In [ ]:
def rag_hyde(pregunta: str, n: int = 2) -> str:
    """
    HyDE: generar documento hipotético con Claude → embeddings → buscar documentos reales similares.
    Idea: el embedding de una respuesta hipotética busca mejor que el embedding de la pregunta.
    """
    # Paso 1: Claude genera un documento hipotético que respondería la pregunta
    doc_hipotetico = client.messages.create(
        model=MODELO, max_tokens=150,
        messages=[{"role": "user", "content": f"Escribe un párrafo técnico breve que responda esta pregunta (sin decir que es hipotético): {pregunta}"}]
    ).content[0].text

    # Paso 2: Buscar documentos reales similares al documento hipotético
    emb_hipotetico = modelo_emb.encode([doc_hipotetico]).tolist()
    res = col_padre.query(query_embeddings=emb_hipotetico, n_results=n)
    contexto = "\n".join(res["documents"][0])

    # Paso 3: Generar respuesta final con contexto real
    return client.messages.create(
        model=MODELO, max_tokens=300,
        messages=[{"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"}]
    ).content[0].text

PREGUNTA = "¿Cuáles son los paradigmas del machine learning?"
print(f"Query: '{PREGUNTA}'")
print("\n=== HyDE ===")
print(rag_hyde(PREGUNTA))

## 4. Estrategia 2: Parent-Child chunking

In [ ]:
def rag_parent_child(pregunta: str, n_hijos: int = 3) -> str:
    """
    Parent-Child: buscar chunks hijos precisos → recuperar artículo padre completo como contexto.
    Mejora: chunks pequeños = mejor precisión en búsqueda; artículo padre = más contexto.
    """
    # Paso 1: buscar chunks hijos precisos
    emb_query = modelo_emb.encode([pregunta]).tolist()
    res_hijos = col_hijos.query(query_embeddings=emb_query, n_results=n_hijos)

    # Paso 2: recuperar artículos padre únicos (sin repetir)
    padre_ids_usados = set()
    contextos_padre = []
    for meta in res_hijos["metadatas"][0]:
        padre_id = meta["padre_id"]
        if padre_id not in padre_ids_usados:
            padre = next((a for a in ARTICULOS_PADRE if a["id"] == padre_id), None)
            if padre:
                contextos_padre.append(f"[{padre['titulo']}]\n{padre['texto']}")
                padre_ids_usados.add(padre_id)

    contexto = "\n\n".join(contextos_padre)

    return client.messages.create(
        model=MODELO, max_tokens=300,
        messages=[{"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"}]
    ).content[0].text

print("\n=== PARENT-CHILD ===")
print(rag_parent_child(PREGUNTA))

## 5. Estrategia 3: Self-Query y comparativa final

In [ ]:
def rag_self_query(pregunta: str) -> str:
    """
    Self-Query: Claude convierte la pregunta en filtros de metadatos + query semántica.
    Permite combinar búsqueda semántica con restricciones estructuradas.
    """
    # Claude extrae filtros de metadatos
    extraccion = client.messages.create(
        model=MODELO, max_tokens=200,
        messages=[{"role": "user", "content": f"""Extrae filtros de metadatos para buscar en una base de datos con campos 'categoria' (ia/datos) y 'nivel' (basico/intermedio/avanzado).

Pregunta: "{pregunta}"

Responde SOLO con JSON: {{"query_semantica": "<consulta simplificada>", "categoria": "<ia|datos|null>", "nivel": "<basico|intermedio|avanzado|null>"}}"""}]
    ).content[0].text.strip()

    if "```" in extraccion:
        extraccion = extraccion.split("```")[1].lstrip("json").strip()

    filtros_extraidos = json.loads(extraccion)
    print(f"  Filtros extraídos: {filtros_extraidos}")

    # Construir filtro de ChromaDB
    where = {}
    if filtros_extraidos.get("categoria") and filtros_extraidos["categoria"] != "null":
        where["categoria"] = filtros_extraidos["categoria"]
    if filtros_extraidos.get("nivel") and filtros_extraidos["nivel"] != "null":
        where["nivel"] = filtros_extraidos["nivel"]

    query_s = filtros_extraidos.get("query_semantica", pregunta)
    emb = modelo_emb.encode([query_s]).tolist()
    kwargs = {"query_embeddings": emb, "n_results": 2}
    if where:
        kwargs["where"] = where

    res = col_padre.query(**kwargs)
    contexto = "\n".join(res["documents"][0])

    return client.messages.create(
        model=MODELO, max_tokens=300,
        messages=[{"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"}]
    ).content[0].text

print("\n=== SELF-QUERY ===")
pregunta_filtrada = "¿Cuáles son los conceptos avanzados de redes neuronales?"
print(rag_self_query(pregunta_filtrada))

print("\n=== CUÁNDO USAR CADA ESTRATEGIA ===")
print("HyDE:          Cuando la query es ambigua o muy corta")
print("Parent-Child:  Cuando necesitas balance entre precisión y contexto")
print("Self-Query:    Cuando tienes metadatos estructurados relevantes")